In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

## Connect to Google Drive

In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab: skipping Drive mount.")

Mounted at /content/drive


## Initialize Constants

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/FDL Project"
DATA_DIR = f"{PROJECT_DIR}/data/flower_photos"
SPLIT_DIR = f"{PROJECT_DIR}/data/splits"

## Split Dataset

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

In [11]:
TEMP_DIR = "/home/matthew/code/notebooks/fdl/FDL Project-20260625T140700Z-3-001/FDL Project/data/flower_photos"

In [12]:
def create_dataset_csvs(data_dir, output_dir=".", n_splits=5, test_size=0.2, random_state=42):
    image_paths = []
    labels = []

    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

    for class_name in classes:
        class_path = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                relative_path = os.path.join(class_name, img_name)
                image_paths.append(relative_path)
                labels.append(class_name)

    full_image_paths = [os.path.join(TEMP_DIR, p) for p in image_paths]

    df = pd.DataFrame({
        'image_path': full_image_paths,
        'label': labels
    })

    print(f"Total images found: {len(df)}")

    class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}

    print("\n=== CLASS LABEL LEGEND ===")
    for class_name, idx in class_to_idx.items():
        print(f"  Class: {class_name:<15} ---> Numeric Label: {idx}")
    print("==========================\n")

    df['label'] = df['label'].map(class_to_idx)

    train_dev_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['label'],
        random_state=random_state
    )

    train_dev_df = train_dev_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    train_dev_df['fold'] = -1

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X=train_dev_df, y=train_dev_df['label'])):
        train_dev_df.loc[val_idx, 'fold'] = fold

    os.makedirs(output_dir, exist_ok=True)

    test_path = os.path.join(output_dir, 'test_local.csv')
    train_folds_path = os.path.join(output_dir, 'train_folds_local.csv')

    test_df.to_csv(test_path, index=False)
    train_dev_df.to_csv(train_folds_path, index=False)

    print("--- CSV Generation Complete ---")
    print(f"Saved {len(test_df)} test samples to: {test_path}")
    print(f"Saved {len(train_dev_df)} cross-val samples to: {train_folds_path}")

In [13]:
# WARNING : EVERYTIME WE CHANGE SEED, WE HAVE TO RE-TRAIN THE MODEL
SEED = 42

if not os.path.exists(DATA_DIR):
    print(f"'{DATA_DIR}' not found.")
else:
    create_dataset_csvs(data_dir=DATA_DIR,
                        output_dir=SPLIT_DIR,
                        n_splits=5,
                        test_size=0.2,
                        random_state = SEED)

Total images found: 3665

=== CLASS LABEL LEGEND ===
  Class: tulips          ---> Numeric Label: 0
  Class: dandelion       ---> Numeric Label: 1
  Class: daisy           ---> Numeric Label: 2
  Class: sunflowers      ---> Numeric Label: 3
  Class: roses           ---> Numeric Label: 4

--- CSV Generation Complete ---
Saved 733 test samples to: /content/drive/MyDrive/FDL Project/data/splits/test_local.csv
Saved 2932 cross-val samples to: /content/drive/MyDrive/FDL Project/data/splits/train_folds_local.csv
